# Batch GPS synchronization of AudioMoth WAV files

Run all cells to process every path in `valid_wav_files`. Each WAV is paired with a same-stem CSV in its own directory (case-insensitive extension). Outputs are saved beside their inputs as `<stem>_PYTHON_SYNC.WAV`.

The synchronization calculations are retained from the original notebook. The defaults still select **2026-06-04 20:30:00**, produce **1,680 seconds**, resolve GPS issues, and extrapolate an unanchored tail using the final 300 anchored seconds. Edit the configuration for other recordings. Missing buffers still fail validation. Outputs are mono 16-bit PCM and omit AudioMoth metadata.

Files are processed sequentially, with independent timing state. Existing outputs are skipped by default. Failures are recorded and processing continues; a completed output is published only after writing succeeds.

In [1]:
import matplotlib.pyplot as plt
import datetime as dt
import pandas as pd
import numpy as np
import scipy as sp

from pathlib import Path


In [2]:
HARD_DRIVE_LOC = Path('/Volumes/Elements')
experiment_folder = HARD_DRIVE_LOC / 'recover-20260604'
RECORDING_TIME = dt.datetime(2026, 6, 4, 20, 30, 0)  # None selects every timestamp
TARGET_OUTPUT_DURATION_SECONDS = 1680
TAIL_RATE_ESTIMATION_SECONDS = 300
RESOLVE_GPS = True
OVERWRITE_EXISTING = False

all_wav_files = sorted(
    path for path in experiment_folder.glob('STF_*/*')
    if path.is_file() and path.suffix.lower() == '.wav'
    and not path.stem.upper().endswith('_SYNC')
)
valid_wav_files = []
for wav_file in all_wav_files:
    try:
        recording_time = dt.datetime.strptime(wav_file.stem, '%Y%m%d_%H%M%S')
    except ValueError:
        continue
    if RECORDING_TIME is None or recording_time == RECORDING_TIME:
        valid_wav_files.append(wav_file)

print(f'Found {len(valid_wav_files)} WAV files to process.')
valid_wav_files

Found 7 WAV files to process.


[PosixPath('/Volumes/Elements/recover-20260604/STF_021/20260604_203000.WAV'),
 PosixPath('/Volumes/Elements/recover-20260604/STF_025/20260604_203000.WAV'),
 PosixPath('/Volumes/Elements/recover-20260604/STF_027/20260604_203000.WAV'),
 PosixPath('/Volumes/Elements/recover-20260604/STF_030/20260604_203000.WAV'),
 PosixPath('/Volumes/Elements/recover-20260604/STF_060/20260604_203000.WAV'),
 PosixPath('/Volumes/Elements/recover-20260604/STF_113/20260604_203000.WAV'),
 PosixPath('/Volumes/Elements/recover-20260604/STF_116/20260604_203000.WAV')]

In [3]:
import math
import wave
import warnings

from scipy.io import wavfile

MICROSECONDS_PER_SECOND = 1_000_000.0
MILLISECONDS_PER_SECOND = 1_000.0

MAX_HFXO_ERROR_ABSOLUTE = 1000 / 1_000_000
MAX_HFXO_ERROR_RELATIVE = 40 / 1_000_000
MAX_LFXO_ERROR = 100 / 1_000_000

CLOCK_DIVIDER = 4
CONVERSION_CYCLES = 12
ACQUISITION_CYCLES = 16
CLOCK_FREQUENCY = 48_000_000
MAXIMUM_REFERENCE_SAMPLE_RATE = 384_000
MAXIMUM_ALLOWABLE_SAMPLE_RATE = 192_000

PPS_CLOCK_TICK_OFFSET = 2 + CLOCK_DIVIDER
MAXIMUM_ALLOWABLE_PPS_OFFSET = (
    PPS_CLOCK_TICK_OFFSET / CLOCK_FREQUENCY * MICROSECONDS_PER_SECOND
)

NUMBER_OF_FIRMWARE_BUFFERS = 8
FIRMWARE_BUFFER_SIZE_BYTES = 32 * 1024
BYTES_PER_SAMPLE = 2


def js_round(value):
    """Match JavaScript Math.round, including its behavior for negative halves."""
    return int(math.floor(float(value) + 0.5))


def calculate_interval_sample_rate(interval):
    """Infer the ADC sample rate between two accepted GPS PPS events."""
    usable_time_us = (
        interval["time_interval_seconds"] * MICROSECONDS_PER_SECOND
        - interval["first_sample_gap_us"]
        - interval["last_sample_gap_us"]
    )
    interval["sample_rate"] = (
        (interval["number_of_samples"] - 1)
        * MICROSECONDS_PER_SECOND
        / usable_time_us
    )
    return interval["sample_rate"]


## Synchronize one recording

Each call loads and validates its own inputs, rebuilds timing intervals, and writes through a temporary file. The nested resampling helpers use only this recording’s timing state.

In [4]:
def synchronize_recording(raw_wav_path, sync_csv_path, output_path):
    required_columns = ["PPS_NUMBER", "AUDIOMOTH_TIME", "TOTAL_SAMPLES",
                        "TIMER_COUNT", "BUFFERS_FILLED", "BUFFERS_WRITTEN"]

    sync_data = pd.read_csv(sync_csv_path)
    missing_columns = sorted(set(required_columns) - set(sync_data.columns))
    assert not missing_columns, f"CSV is missing required columns: {missing_columns}"
    assert len(sync_data) >= 2, "The CSV must contain at least two PPS events."

    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        source_sample_rate, raw_samples = wavfile.read(raw_wav_path, mmap=True)

    assert raw_samples.ndim == 1, "The AudioMoth GPS Sync WAV must be mono."
    assert raw_samples.dtype == np.int16, (f"Expected signed 16-bit PCM, got {raw_samples.dtype}.")

    sync_data["audiomoth_datetime"] = pd.to_datetime(sync_data["AUDIOMOTH_TIME"], utc=True, errors="raise")
    sync_data["audiomoth_time_ms"] = sync_data["audiomoth_datetime"].astype("int64") // 1_000_000

    filename_datetime = pd.to_datetime(raw_wav_path.stem, format="%Y%m%d_%H%M%S", utc=True)
    first_timestamp_difference_ms = abs((sync_data["audiomoth_datetime"].iloc[0] - filename_datetime).total_seconds() * 1000)
    assert first_timestamp_difference_ms <= 500, (
        f"The first CSV AudioMoth timestamp differs from the WAV filename by {first_timestamp_difference_ms:.3f} ms.")

    print(f"Raw WAV: {raw_wav_path}")
    print(f"Sync CSV: {sync_csv_path}")
    print(f"Nominal sample rate: {source_sample_rate:,} Hz")
    print(f"Raw samples: {len(raw_samples):,}")
    print(f"PPS rows: {len(sync_data):,}")


    buffer_overflows = []
    previous_skipped_buffers = 0

    for row_index in range(1, len(sync_data)):
        buffers_filled = int(sync_data["BUFFERS_FILLED"].iloc[row_index])
        buffers_written = int(sync_data["BUFFERS_WRITTEN"].iloc[row_index])
        difference = buffers_filled - buffers_written - previous_skipped_buffers

        if difference >= NUMBER_OF_FIRMWARE_BUFFERS:
            skipped = NUMBER_OF_FIRMWARE_BUFFERS * (
                difference // NUMBER_OF_FIRMWARE_BUFFERS
            )
            missing_samples = (
                FIRMWARE_BUFFER_SIZE_BYTES // BYTES_PER_SAMPLE * skipped
            )
            previous_skipped_buffers += skipped
            buffer_overflows.append(
                {
                    "csv_row": row_index,
                    "skipped_buffers": skipped,
                    "missing_samples": missing_samples,
                }
            )

    assert not buffer_overflows, (
        "This file contains missing WAV buffers. Choose another valid pair or "
        "implement the application's resolve-WAV zero insertion option. "
        f"Events: {buffer_overflows[:5]}"
    )
    print("No dropped firmware buffers detected.")



    sample_interval_us = MICROSECONDS_PER_SECOND / source_sample_rate
    oversample_rate = 2 ** math.floor(
        math.log2(MAXIMUM_REFERENCE_SAMPLE_RATE / source_sample_rate)
    )
    clock_ticks_between_samples = CLOCK_FREQUENCY / source_sample_rate
    clock_ticks_to_complete_sample = (
        2
        + CLOCK_DIVIDER
        * (2 + oversample_rate * (ACQUISITION_CYCLES + CONVERSION_CYCLES))
    )

    timer_count = sync_data["TIMER_COUNT"].to_numpy(dtype=np.float64)
    time_to_next_sample_us = np.where(
        timer_count <= clock_ticks_to_complete_sample,
        clock_ticks_to_complete_sample - timer_count,
        clock_ticks_between_samples + clock_ticks_to_complete_sample - timer_count,
    ) / CLOCK_FREQUENCY * MICROSECONDS_PER_SECOND

    total_samples_at_pps = sync_data["TOTAL_SAMPLES"].to_numpy(dtype=np.int64)
    time_at_pps_ms = sync_data["audiomoth_time_ms"].to_numpy(dtype=np.int64)

    intervals = []
    rejected_pps_rows = []
    cumulative_time_seconds = 0
    current_index = 0
    current_total_samples = int(total_samples_at_pps[0])
    current_time_ms = int(time_at_pps_ms[0])
    sample_rate_total = 0
    sample_rate_count = 0

    for next_index in range(1, len(sync_data)):
        next_total_samples = int(total_samples_at_pps[next_index])
        next_time_ms = int(time_at_pps_ms[next_index])

        number_of_samples = next_total_samples - current_total_samples
        measured_interval_ms = next_time_ms - current_time_ms
        rounded_interval_seconds = js_round(
            measured_interval_ms / MILLISECONDS_PER_SECOND
        )

        target_rate = (
            source_sample_rate
            if sample_rate_count == 0
            else sample_rate_total / sample_rate_count
        )
        maximum_time_error_ms = math.ceil(
            MAX_LFXO_ERROR
            * rounded_interval_seconds
            * MILLISECONDS_PER_SECOND
        )
        maximum_sample_error = math.ceil(
            (
                MAX_HFXO_ERROR_ABSOLUTE
                if sample_rate_count == 0
                else MAX_HFXO_ERROR_RELATIVE
            )
            * (source_sample_rate if sample_rate_count == 0 else target_rate)
            * rounded_interval_seconds
        )

        time_error_ms = abs(measured_interval_ms - rounded_interval_seconds * MILLISECONDS_PER_SECOND)
        sample_error = abs(number_of_samples - rounded_interval_seconds * target_rate)
        valid_interval = rounded_interval_seconds > 0 and time_error_ms <= maximum_time_error_ms and sample_error <= maximum_sample_error

        if not valid_interval:
            rejected_pps_rows.append(next_index)
            if not RESOLVE_GPS:
                raise ValueError(
                    "Misaligned PPS event at CSV row "
                    f"{next_index}: {number_of_samples} samples over {measured_interval_ms:.3f} ms; "
                    f"time error {time_error_ms:.6f}/{maximum_time_error_ms} ms, "
                    f"sample error {sample_error:.3f}/{maximum_sample_error} samples."
                )
            continue

        if rounded_interval_seconds > 1 and not RESOLVE_GPS:
            raise ValueError(
                f"Missing PPS event before CSV row {next_index}; "
                f"accepted interval is {rounded_interval_seconds} seconds."
            )

        sample_rate_total += number_of_samples
        sample_rate_count += rounded_interval_seconds

        interval = {
            "interval_index": len(intervals),
            "start_pps_index": current_index,
            "end_pps_index": next_index,
            "time_interval_seconds": rounded_interval_seconds,
            "cumulative_time_seconds": cumulative_time_seconds,
            "number_of_samples": number_of_samples,
            "first_sample_gap_us": float(time_to_next_sample_us[current_index]),
            "last_sample_gap_us": float(
                sample_interval_us - time_to_next_sample_us[next_index]
            ),
            "extrapolated": False,
        }
        calculate_interval_sample_rate(interval)
        intervals.append(interval)

        cumulative_time_seconds += rounded_interval_seconds
        current_index = next_index
        current_total_samples = next_total_samples
        current_time_ms = next_time_ms

    assert intervals, "No valid interval exists between PPS events."

    average_sample_rate = (
        sum(x["number_of_samples"] for x in intervals)
        / sum(x["time_interval_seconds"] for x in intervals)
    )

    print(f"Accepted intervals: {len(intervals):,}")
    print(f"Rejected PPS rows: {rejected_pps_rows}")
    print(f"Average raw rate: {average_sample_rate:.6f} samples/s")


    # Correct PPS/sample ordering ambiguities.
    for interval_index in range(len(intervals) - 1):
        interval = intervals[interval_index]
        next_interval = intervals[interval_index + 1]

        if (
            interval["last_sample_gap_us"] < MAXIMUM_ALLOWABLE_PPS_OFFSET
            and js_round(interval["sample_rate"] - average_sample_rate) == -1
            and js_round(next_interval["sample_rate"] - average_sample_rate) == 1
        ):
            interval["last_sample_gap_us"] = sample_interval_us
            calculate_interval_sample_rate(interval)
            next_interval["first_sample_gap_us"] = 0.0
            calculate_interval_sample_rate(next_interval)

    if source_sample_rate == MAXIMUM_ALLOWABLE_SAMPLE_RATE:
        first_interval = intervals[0]
        if js_round(first_interval["sample_rate"] - average_sample_rate) == 1:
            first_interval["first_sample_gap_us"] -= sample_interval_us
            calculate_interval_sample_rate(first_interval)

        for interval_index in range(len(intervals) - 1):
            interval = intervals[interval_index]
            next_interval = intervals[interval_index + 1]
            if (
                interval["last_sample_gap_us"] < MAXIMUM_ALLOWABLE_PPS_OFFSET
                and js_round(interval["sample_rate"] - average_sample_rate) == -1
                and js_round(next_interval["sample_rate"] - average_sample_rate) == 0
            ):
                interval["last_sample_gap_us"] = sample_interval_us
                calculate_interval_sample_rate(interval)
                next_interval["first_sample_gap_us"] = sample_interval_us
                calculate_interval_sample_rate(next_interval)

        for interval in intervals:
            if js_round(interval["sample_rate"] - average_sample_rate) == -1:
                interval["first_sample_gap_us"] += sample_interval_us
                calculate_interval_sample_rate(interval)

    first_sample_is_before_first_interval = (
        intervals[0]["first_sample_gap_us"] < 0
    )

    # Align samples to the midpoint of the ADC acquisition period.
    last_acquisition_ends = 1 + CLOCK_DIVIDER * (CONVERSION_CYCLES + 1)
    first_acquisition_starts = (
        clock_ticks_to_complete_sample - 1 - CLOCK_DIVIDER
    )
    adc_time_offset_us = (
        MICROSECONDS_PER_SECOND
        * (last_acquisition_ends + first_acquisition_starts)
        / 2
        / CLOCK_FREQUENCY
    )

    for interval_index, interval in enumerate(intervals):
        interval["first_sample_gap_us"] -= adc_time_offset_us
        interval["last_sample_gap_us"] += adc_time_offset_us

        if interval["first_sample_gap_us"] < 0:
            interval["number_of_samples"] -= 1
            interval["first_sample_gap_us"] += sample_interval_us
            if interval_index == 0:
                first_sample_is_before_first_interval = True
            else:
                previous_interval = intervals[interval_index - 1]
                previous_interval["number_of_samples"] += 1
                previous_interval["last_sample_gap_us"] = (
                    sample_interval_us - interval["first_sample_gap_us"]
                )

    for interval in intervals:
        calculate_interval_sample_rate(interval)

    unusual_intervals = [
        interval["interval_index"]
        for interval in intervals
        if js_round(interval["sample_rate"] - average_sample_rate) != 0
    ]
    if unusual_intervals and not RESOLVE_GPS:
        raise ValueError(
            "Unusual sample count remains in intervals "
            f"{unusual_intervals[:20]}."
        )

    # The firmware's 192 kHz mode misses the first sample at each interval.
    if source_sample_rate == MAXIMUM_ALLOWABLE_SAMPLE_RATE:
        for interval in intervals:
            interval["first_sample_gap_us"] += sample_interval_us
            calculate_interval_sample_rate(interval)

    # Repair a final section for which recording samples exist but PPS anchors do not.
    anchored_duration_seconds = sum(interval["time_interval_seconds"] for interval in intervals)
    seconds_to_extrapolate = TARGET_OUTPUT_DURATION_SECONDS - anchored_duration_seconds

    if seconds_to_extrapolate < 0:
        raise ValueError(f"The accepted GPS intervals already exceed the requested {TARGET_OUTPUT_DURATION_SECONDS}-second output.")

    if seconds_to_extrapolate > 0:
        recent_intervals = []
        recent_duration_seconds = 0
        for interval in reversed(intervals):
            recent_intervals.append(interval)
            recent_duration_seconds += interval["time_interval_seconds"]
            if recent_duration_seconds >= TAIL_RATE_ESTIMATION_SECONDS:
                break

        estimated_tail_rate = sum(interval["number_of_samples"] for interval in recent_intervals) / sum(interval["time_interval_seconds"] for interval in recent_intervals)
        extrapolated_sample_count = js_round(seconds_to_extrapolate * estimated_tail_rate)
        extrapolated_first_gap_us = float(time_to_next_sample_us[current_index])
        extrapolated_last_gap_us = seconds_to_extrapolate * MICROSECONDS_PER_SECOND - extrapolated_first_gap_us - (extrapolated_sample_count - 1) * MICROSECONDS_PER_SECOND / estimated_tail_rate

        # Apply the same ADC midpoint alignment used for measured intervals.
        extrapolated_first_gap_us -= adc_time_offset_us
        extrapolated_last_gap_us += adc_time_offset_us
        if extrapolated_first_gap_us < 0:
            extrapolated_sample_count -= 1
            extrapolated_first_gap_us += sample_interval_us
            intervals[-1]["number_of_samples"] += 1
            intervals[-1]["last_sample_gap_us"] = sample_interval_us - extrapolated_first_gap_us
            calculate_interval_sample_rate(intervals[-1])

        if source_sample_rate == MAXIMUM_ALLOWABLE_SAMPLE_RATE:
            extrapolated_first_gap_us += sample_interval_us

        extrapolated_interval = {
            "interval_index": len(intervals),
            "start_pps_index": current_index,
            "end_pps_index": None,
            "time_interval_seconds": seconds_to_extrapolate,
            "cumulative_time_seconds": anchored_duration_seconds,
            "number_of_samples": extrapolated_sample_count,
            "first_sample_gap_us": extrapolated_first_gap_us,
            "last_sample_gap_us": extrapolated_last_gap_us,
            "extrapolated": True,
        }
        calculate_interval_sample_rate(extrapolated_interval)
        intervals.append(extrapolated_interval)

    raw_samples_used = sum(interval["number_of_samples"] for interval in intervals)
    discarded_raw_samples = len(raw_samples) - raw_samples_used
    assert discarded_raw_samples >= 0, f"The repaired intervals require {-discarded_raw_samples:,} more raw samples than the WAV contains."
    assert sum(interval["time_interval_seconds"] for interval in intervals) == TARGET_OUTPUT_DURATION_SECONDS

    if seconds_to_extrapolate > 0:
        print(f"Extrapolated final {seconds_to_extrapolate} seconds at {estimated_tail_rate:.6f} raw samples/s")
        print(f"Raw samples intentionally discarded after scheduled endpoint: {discarded_raw_samples:,} ({discarded_raw_samples / estimated_tail_rate:.6f} s)")

    interval_table = pd.DataFrame(intervals)
    interval_columns = [
            "interval_index",
            "start_pps_index",
            "end_pps_index",
            "time_interval_seconds",
            "number_of_samples",
            "sample_rate",
            "first_sample_gap_us",
            "last_sample_gap_us",
            "extrapolated",
    ]
    def round_and_clip_int16(values):
        """Match Math.round and convert interpolated values to 16-bit PCM."""
        rounded = np.floor(values + 0.5)
        return np.clip(rounded, -32768, 32767).astype("<i2")


    def resample_one_interval(
        samples,
        first_input_index,
        interval,
        target_sample_rate,
        algorithm="linear",
    ):
        """Return one GPS-aligned interval and the next raw sample index."""
        input_count = int(interval["number_of_samples"])
        final_input_index = first_input_index + input_count
        if final_input_index >= len(samples):
            raise ValueError(
                "The CSV asks for more samples than are present in the raw WAV: "
                f"needed index {final_input_index:,}, "
                f"last available index is {len(samples) - 1:,}."
            )

        if first_input_index == 0:
            # The official stream starts with previous == next == sample 0.
            # Duplicate that value so it has both the previous- and next-sample
            # timestamps during the first interval.
            input_indices = np.concatenate(
                (
                    np.array([0], dtype=np.int64),
                    np.arange(0, final_input_index + 1, dtype=np.int64),
                )
            )
        else:
            input_indices = np.arange(
                first_input_index - 1,
                final_input_index + 1,
                dtype=np.int64,
            )
        input_values = np.asarray(samples[input_indices], dtype=np.float64)

        first_gap_seconds = (
            interval["first_sample_gap_us"] / MICROSECONDS_PER_SECOND
        )
        previous_gap_seconds = first_gap_seconds - (
            sample_interval_us / MICROSECONDS_PER_SECOND
        )
        input_times = np.empty(len(input_indices), dtype=np.float64)
        input_times[0] = previous_gap_seconds
        input_times[1:] = first_gap_seconds + (
            np.arange(len(input_indices) - 1, dtype=np.float64)
            / interval["sample_rate"]
        )

        number_of_output_samples = (
            int(interval["time_interval_seconds"]) * target_sample_rate
        )
        output_times = (
            np.arange(number_of_output_samples, dtype=np.float64)
            / target_sample_rate
        )

        if algorithm == "linear":
            output_values = np.interp(output_times, input_times, input_values)
        elif algorithm == "nearest":
            right = np.searchsorted(input_times, output_times, side="left")
            right = np.clip(right, 1, len(input_times) - 1)
            left = right - 1
            choose_right = (
                input_times[right] - output_times
                < 0.5 / interval["sample_rate"]
            )
            selected = np.where(choose_right, right, left)
            output_values = input_values[selected]
        else:
            raise ValueError("algorithm must be 'linear' or 'nearest'")

        return round_and_clip_int16(output_values), final_input_index


    def write_gps_synchronized_wav(
        samples,
        intervals,
        source_rate,
        output_path,
        target_rate=None,
        algorithm="linear",
    ):
        """Write a mono 16-bit PCM WAV on an exact GPS-spaced sample grid."""
        output_path = Path(output_path)
        output_path.parent.mkdir(parents=True, exist_ok=True)
        target_rate = source_rate if target_rate is None else int(target_rate)

        if target_rate < source_rate:
            raise ValueError(
                "The AudioMoth sync app does not permit a target rate below "
                "the raw WAV sample rate."
            )

        first_input_index = 1 if first_sample_is_before_first_interval else 0
        total_output_samples = 0

        with wave.open(str(output_path), "wb") as output_wav:
            output_wav.setnchannels(1)
            output_wav.setsampwidth(BYTES_PER_SAMPLE)
            output_wav.setframerate(target_rate)

            for interval_index, interval in enumerate(intervals):
                aligned_samples, first_input_index = resample_one_interval(
                    samples=samples,
                    first_input_index=first_input_index,
                    interval=interval,
                    target_sample_rate=target_rate,
                    algorithm=algorithm,
                )
                output_wav.writeframesraw(aligned_samples.tobytes())
                total_output_samples += len(aligned_samples)

                if (
                    (interval_index + 1) % 60 == 0
                    or interval_index + 1 == len(intervals)
                ):
                    print(
                        f"Processed {interval_index + 1:,}/{len(intervals):,} "
                        "PPS intervals"
                    )

        expected_output_samples = (
            sum(x["time_interval_seconds"] for x in intervals) * target_rate
        )
        assert total_output_samples == expected_output_samples
        return output_path

    first_input_index = 1 if first_sample_is_before_first_interval else 0
    required_last_index = first_input_index + sum(x['number_of_samples'] for x in intervals)
    if required_last_index >= len(raw_samples):
        raise ValueError('Insufficient raw samples for interpolation at the output endpoint.')

    import os
    import tempfile
    fd, temporary_name = tempfile.mkstemp(prefix=output_path.stem + '_', suffix='.tmp', dir=output_path.parent)
    os.close(fd)
    temporary_path = Path(temporary_name)
    try:
        write_gps_synchronized_wav(
            samples=raw_samples, intervals=intervals, source_rate=source_sample_rate,
            output_path=temporary_path, target_rate=source_sample_rate, algorithm='linear',
        )
        temporary_path.replace(output_path)
    finally:
        temporary_path.unlink(missing_ok=True)

    return {
        'sample_rate': source_sample_rate,
        'duration_seconds': sum(x['time_interval_seconds'] for x in intervals),
        'extrapolated_seconds': seconds_to_extrapolate,
        'rejected_pps_count': len(rejected_pps_rows),
    }

## Process every valid WAV

Inspect `batch_results` after completion for written, skipped, and failed files. Correct failed inputs and rerun this cell; completed outputs are skipped unless `OVERWRITE_EXISTING` is enabled.

In [6]:
def matching_csv(raw_wav_path):
    candidates = sorted(
        path for path in raw_wav_path.parent.iterdir()
        if path.is_file() and path.stem == raw_wav_path.stem
        and path.suffix.lower() == '.csv'
    )
    if len(candidates) != 1:
        raise ValueError(
            f'Expected one matching CSV for {raw_wav_path}; found {len(candidates)}: {candidates}'
        )
    return candidates[0]

results = []
for index, raw_wav_path in enumerate(valid_wav_files, start=1):
    raw_wav_path = Path(raw_wav_path)
    output_path = raw_wav_path.with_name(f'{raw_wav_path.stem}_PYTHON_SYNC.WAV')
    result = {'raw_wav': str(raw_wav_path), 'sync_csv': None,
              'output_wav': str(output_path), 'status': None, 'error': None}
    print(f'\n[{index}/{len(valid_wav_files)}] {raw_wav_path}')
    try:
        sync_csv_path = matching_csv(raw_wav_path)
        result['sync_csv'] = str(sync_csv_path)
        result.update(synchronize_recording(raw_wav_path, sync_csv_path, output_path))
        result['status'] = 'written'
        print(f'Wrote: {output_path}')
    except Exception as exc:
        result['status'] = 'failed'
        result['error'] = f'{type(exc).__name__}: {exc}'
        print(f"FAILED: {result['error']}")
    results.append(result)

batch_results = pd.DataFrame(results, columns=[
    'raw_wav', 'sync_csv', 'output_wav', 'status', 'error',
    'sample_rate', 'duration_seconds', 'extrapolated_seconds', 'rejected_pps_count',
])
print(batch_results['status'].value_counts().to_string() if results else 'No matching WAV files found; check the folder and RECORDING_TIME.')
display(batch_results)


[1/7] /Volumes/Elements/recover-20260604/STF_021/20260604_203000.WAV
Raw WAV: /Volumes/Elements/recover-20260604/STF_021/20260604_203000.WAV
Sync CSV: /Volumes/Elements/recover-20260604/STF_021/20260604_203000.CSV
Nominal sample rate: 192,000 Hz
Raw samples: 322,560,000
PPS rows: 1,326
No dropped firmware buffers detected.
Accepted intervals: 1,325
Rejected PPS rows: []
Average raw rate: 191966.326621 samples/s
Extrapolated final 91 seconds at 191966.306667 raw samples/s
Raw samples intentionally discarded after scheduled endpoint: 56,574 (0.294708 s)
Processed 60/1,326 PPS intervals
Processed 120/1,326 PPS intervals
Processed 180/1,326 PPS intervals
Processed 240/1,326 PPS intervals
Processed 300/1,326 PPS intervals
Processed 360/1,326 PPS intervals
Processed 420/1,326 PPS intervals
Processed 480/1,326 PPS intervals
Processed 540/1,326 PPS intervals
Processed 600/1,326 PPS intervals
Processed 660/1,326 PPS intervals
Processed 720/1,326 PPS intervals
Processed 780/1,326 PPS intervals


,raw_wav,sync_csv,output_wav,status,error,sample_rate,duration_seconds,extrapolated_seconds,rejected_pps_count
0,/Volumes/Elements/recover-20260604/STF_021/202...,/Volumes/Elements/recover-20260604/STF_021/202...,/Volumes/Elements/recover-20260604/STF_021/202...,written,None,192000,1680,91,0
1,/Volumes/Elements/recover-20260604/STF_025/202...,/Volumes/Elements/recover-20260604/STF_025/202...,/Volumes/Elements/recover-20260604/STF_025/202...,written,None,192000,1680,0,0
2,/Volumes/Elements/recover-20260604/STF_027/202...,/Volumes/Elements/recover-20260604/STF_027/202...,/Volumes/Elements/recover-20260604/STF_027/202...,written,None,192000,1680,0,0
3,/Volumes/Elements/recover-20260604/STF_030/202...,/Volumes/Elements/recover-20260604/STF_030/202...,/Volumes/Elements/recover-20260604/STF_030/202...,written,None,192000,1680,0,0
4,/Volumes/Elements/recover-20260604/STF_060/202...,/Volumes/Elements/recover-20260604/STF_060/202...,/Volumes/Elements/recover-20260604/STF_060/202...,written,None,192000,1680,0,0
5,/Volumes/Elements/recover-20260604/STF_113/202...,/Volumes/Elements/recover-20260604/STF_113/202...,/Volumes/Elements/recover-20260604/STF_113/202...,written,None,192000,1680,0,0
6,/Volumes/Elements/recover-20260604/STF_116/202...,/Volumes/Elements/recover-20260604/STF_116/202...,/Volumes/Elements/recover-20260604/STF_116/202...,written,None,192000,1680,0,0
